# Overfit One SST-2 Activation Oracle Sample (arena-env)

Same as `overfit_single_sst2_sample.ipynb` but works with the `arena-env` conda environment.
Two changes from the original:
1. `train_features_batch` is inlined — avoids the `slist` transitive import via `nl_probes.sft → sae_training_data → caller`
2. `attn_implementation='sdpa'` — `flash_attention_2` is not in arena-env

In [1]:
from pathlib import Path
import os
import random
import sys

import torch
from peft import LoraConfig, PeftModel, get_peft_model
from torch.nn.utils import clip_grad_norm_
from transformers import BitsAndBytesConfig

THIS_DIR = Path.cwd()
WORKSPACE_ROOT = THIS_DIR.parent
REPO_DIR = WORKSPACE_ROOT / 'activation_oracles'
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)

from nl_probes.configs.sft_config import SelfInterpTrainingConfig
from nl_probes.dataset_classes.act_dataset_manager import DatasetLoaderConfig
from nl_probes.dataset_classes.classification import ClassificationDatasetConfig, ClassificationDatasetLoader
from nl_probes.utils.activation_utils import get_hf_submodule, get_text_only_lora_targets
from nl_probes.utils.common import load_model, load_tokenizer, set_seed
from nl_probes.utils.dataset_utils import construct_batch, materialize_missing_steering_vectors
from nl_probes.utils.steering_hooks import add_hook, get_hf_activation_steering_hook

In [2]:
def train_features_batch(cfg, training_batch, model, submodule, device, dtype):
    """Inlined from nl_probes.sft to avoid the slist transitive import."""
    hook_fn = get_hf_activation_steering_hook(
        vectors=training_batch.steering_vectors,
        positions=training_batch.positions,
        steering_coefficient=cfg.steering_coefficient,
        device=device,
        dtype=dtype,
    )
    tokenized_input = {
        'input_ids': training_batch.input_ids,
        'attention_mask': training_batch.attention_mask,
    }
    with add_hook(submodule, hook_fn):
        loss = model(**tokenized_input, labels=training_batch.labels).loss
    return loss

In [3]:
model_name = 'Qwen/Qwen3-1.7B'
dataset_folder = 'sft_training_data'
output_dir = WORKSPACE_ROOT / 'ao_minimal_training' / 'checkpoints_single_sample_overfit'

layer_percents = [25, 50, 75]
hook_layer = 1
save_acts = False
num_train = 1
seed = 42
lr = 1e-4
num_steps = 100
load_in_8bit = True

set_seed(seed)
device = torch.device('cuda:0')
dtype = torch.bfloat16

In [4]:
params = ClassificationDatasetConfig(
    classification_dataset_name='sst2',
    max_window_size=1,
    min_window_size=1,
    min_end_offset=-1,
    max_end_offset=-5,
    num_qa_per_sample=2,
)
dataset_cfg = DatasetLoaderConfig(
    custom_dataset_params=params,
    num_train=num_train,
    num_test=0,
    splits=['train'],
    model_name=model_name,
    layer_percents=layer_percents,
    save_acts=save_acts,
    batch_size=1,
    dataset_folder=str(dataset_folder),
)
loader = ClassificationDatasetLoader(dataset_config=dataset_cfg, model_kwargs={})

# load_dataset creates the cache automatically if missing (makedirs + create_dataset)
training_data = loader.load_dataset('train')
dp = training_data[0]
print(f'Loaded {len(training_data)} datapoints; using one sample')
print('target_output:', dp.target_output)
print('layer:', dp.layer)
print('num activation positions:', len(dp.positions))

Loaded 6 datapoints from sft_training_data/classification_sst2_model_Qwen3-1_7B_n_1_save_acts_False_train_3d138cd8add4.pt
Loaded 6 datapoints; using one sample
target_output: No
layer: 7
num activation positions: 1


In [5]:
cfg = SelfInterpTrainingConfig(
    model_name=model_name,
    hook_onto_layer=hook_layer,
    layer_percents=layer_percents,
    train_batch_size=1,
    lr=lr,
    steering_coefficient=1.0,
    save_dir=str(output_dir),
    seed=seed,
).finalize(dataset_loaders=[loader])

tokenizer = load_tokenizer(model_name)
model_kwargs = {'device_map': {'': 'cuda:0'}, 'attn_implementation': 'sdpa'}
if load_in_8bit:
    model_kwargs['quantization_config'] = BitsAndBytesConfig(load_in_8bit=True, bnb_8bit_compute_dtype=dtype)

model = load_model(model_name, dtype, **model_kwargs)
model.enable_input_require_grads()

target_modules = cfg.lora_target_modules
vlm_targets = get_text_only_lora_targets(cfg.model_name)
if vlm_targets and target_modules == 'all-linear':
    target_modules = vlm_targets
lora_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=0.0,
    target_modules=target_modules,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config, autocast_adapter_dtype=True)
model.print_trainable_parameters()
model.train()
submodule = get_hf_submodule(model, hook_layer, use_lora=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

📦 Loading tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!


🧠 Loading model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

trainable params: 69,730,304 || all params: 1,790,305,280 || trainable%: 3.8949


In [6]:
losses = []
optimizer.zero_grad()
for step in range(num_steps):
    batch_points = materialize_missing_steering_vectors([dp], tokenizer, model)
    batch = construct_batch(batch_points, tokenizer, device)
    loss = train_features_batch(cfg, batch, model, submodule, device, dtype)
    loss.backward()
    clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
    optimizer.step()
    optimizer.zero_grad()
    losses.append(float(loss.item()))
    print(f'step {step:03d} loss {losses[-1]:.6f}')

/opt/conda/envs/arena-env/lib/python3.11/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


step 000 loss 8.164639
step 001 loss 4.291675
step 002 loss 0.000108
step 003 loss 0.000067
step 004 loss 0.000395
step 005 loss 0.000028
step 006 loss 0.000013
step 007 loss 0.000013
step 008 loss 0.000024
step 009 loss 0.000006
step 010 loss 0.000007
step 011 loss 0.000009
step 012 loss 0.000004
step 013 loss 0.000006
step 014 loss 0.000003
step 015 loss 0.000004
step 016 loss 0.000002
step 017 loss 0.000002
step 018 loss 0.000002
step 019 loss 0.000003
step 020 loss 0.000002
step 021 loss 0.000003
step 022 loss 0.000002
step 023 loss 0.000002
step 024 loss 0.000002
step 025 loss 0.000002
step 026 loss 0.000002
step 027 loss 0.000002
step 028 loss 0.000002
step 029 loss 0.000002
step 030 loss 0.000002
step 031 loss 0.000002
step 032 loss 0.000002
step 033 loss 0.000002
step 034 loss 0.000002
step 035 loss 0.000002
step 036 loss 0.000002


KeyboardInterrupt: 

In [7]:
output_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f'Saved overfit LoRA adapter to {output_dir}')

Saved overfit LoRA adapter to /root/evil_oracles/ao_minimal_training/checkpoints_single_sample_overfit


## Optional generation check

Checks whether the overfit adapter can generate the target answer from the same activation-injected prompt.

In [8]:
from nl_probes.utils.eval import run_evaluation

model.eval()
result = run_evaluation(
    eval_data=[dp],
    model=model,
    tokenizer=tokenizer,
    submodule=submodule,
    device=device,
    dtype=dtype,
    global_step=0,
    lora_path=None,
    eval_batch_size=1,
    steering_coefficient=cfg.steering_coefficient,
    generation_kwargs={'do_sample': False, 'max_new_tokens': 5},
)
print('target:', dp.target_output)
print('generated:', result[0].api_response)

Evaluating model: 100%|██████████| 1/1 [00:00<00:00,  1.21it/s]

target: No
generated: No


In [9]:
print(dp)

datapoint_type='classification_sst2' input_ids=[151644, 872, 198, 9188, 25, 220, 22, 198, 937, 715, 16141, 448, 364, 9454, 6, 476, 364, 2753, 6, 1172, 13, 18885, 498, 1977, 419, 374, 264, 36749, 3395, 30, 151645, 198, 151644, 77091, 198, 151667, 271, 151668, 271, 2753, 151645, 198] labels=[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 2753, 151645, 198] layer=7 steering_vectors=None positions=[8] feature_idx=-1 target_output='No' context_input_ids=[151644, 872, 198, 18691, 3694, 78398, 369, 279, 2097, 429, 432, 374, 7009, 11997, 311, 1401, 518, 476, 3535, 151645, 198] context_positions=[17] ds_label='negative' meta_info={}
